# Whisper for transcription Demo

In [2]:
!pip install -q yt-dlp
!apt-get -y install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 256.3 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 97.6 kB/s eta 0:00:0000:0100:02m
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 138 not upgraded.


In [3]:
import yt_dlp

video_id = "Iot0eF6EoNA"
video_url = f"https://youtube.com/watch?v={video_id}"

ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': '/kaggle/working/audio.%(ext)s',  # important path for Kaggle
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
    }],
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([video_url])

[youtube] Extracting URL: https://youtube.com/watch?v=Iot0eF6EoNA
[youtube] Iot0eF6EoNA: Downloading webpage


[youtube] Iot0eF6EoNA: Downloading android vr player API JSON
[info] Iot0eF6EoNA: Downloading 1 format(s): 251
[download] Destination: /kaggle/working/audio.webm
[download] 100% of    2.68MiB in 00:00:00 at 4.12MiB/s   
[ExtractAudio] Destination: /kaggle/working/audio.mp3
Deleting original file /kaggle/working/audio.webm (pass -k to keep)


In [6]:
!pip install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 45.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 41.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 74.0 MB/s eta 0:00:00:00:0100:01


In [8]:
import os
import wave
import time
import struct
import tempfile
import numpy as np
#import pyaudio
from pathlib import Path
from faster_whisper import WhisperModel

MODEL_SIZE   = "small"    # tiny / base / small / medium / large
DEVICE       = "cpu"     # cpu or cuda
COMPUTE_TYPE = "int8"    # int8 for cpu, float16 for gpu
LANGUAGE     = "en"      # en, hi, ta, etc. or None for auto-detect

SAMPLE_RATE          = 16000  # 16kHz — Whisper's preferred rate
CHUNK_SIZE           = 1024   # audio frames per read

def load_model(model_size=MODEL_SIZE, device=DEVICE,
               compute_type=COMPUTE_TYPE):
    """
    Load faster-whisper model.
    Downloads on first run (~150MB for base), cached after.

    Model size comparison:
        tiny   : fastest, least accurate (~39MB)
        base   : good balance for testing (~74MB)
        small  : better accuracy (~244MB)
        medium : very accurate, slow on CPU (~769MB)
        large  : best accuracy, needs GPU (~1.5GB)
    """
    print(f"\nLoading Whisper {model_size} on {device}...")
    start = time.time()

    model = WhisperModel(
        model_size,
        device=device,
        compute_type=compute_type
    )

    print(f"Model loaded in {time.time()-start:.1f}s")
    return model

# ─────────────────────────────────────────────────────────────
# SECTION 2 — Transcribe from existing file
# ─────────────────────────────────────────────────────────────

def transcribe_file(model, file_path, language=LANGUAGE):
    """
    Transcribe an existing audio or video file.
    Supports: .wav .mp3 .mp4 .webm .m4a .ogg

    Args:
        model     : WhisperModel from load_model()
        file_path : path to audio or video file
        language  : language code or None for auto-detect

    Returns:
        dict with text, segments, language, duration
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    print(f"\nTranscribing file: {file_path.name}")
    print(f"Language: {language or 'auto-detect'}")
    start = time.time()

    # Transcribe
    segments, info = model.transcribe(
        str(file_path),
        language=language,
        beam_size=5,
        vad_filter=True,
        vad_parameters=dict(
            min_silence_duration_ms=500
        )
    )

    # Collect results
    all_segments = []
    full_text    = []

    for segment in segments:
        seg = {
            "start" : round(segment.start, 2),
            "end"   : round(segment.end, 2),
            "text"  : segment.text.strip()
        }
        all_segments.append(seg)
        full_text.append(segment.text.strip())

    elapsed = round(time.time() - start, 2)

    result = {
        "text"       : " ".join(full_text),
        "segments"   : all_segments,
        "language"   : info.language,
        "duration"   : round(info.duration, 2),
        "time_taken" : elapsed
    }

    # Print results
    print(f"\n{'='*50}")
    print(f"TRANSCRIPTION RESULT")
    print(f"{'='*50}")
    print(f"Language detected : {result['language']}")
    print(f"Audio duration    : {result['duration']}s")
    print(f"Time taken        : {result['time_taken']}s")
    print(f"\nFull text:")
    print(f"  {result['text']}")
    print(f"\nSegments (with timestamps):")
    for seg in result['segments']:
        print(f"  [{seg['start']}s → {seg['end']}s] {seg['text']}")
    print(f"{'='*50}")

if __name__ == "__main__":
    file_path = '/kaggle/working/audio.mp3'
    model  = load_model()
    result = transcribe_file(model, file_path)


Loading Whisper small on cpu...
Model loaded in 4.8s

Transcribing file: audio.mp3
Language: en

TRANSCRIPTION RESULT
Language detected : en
Audio duration    : 182.3s
Time taken        : 74.59s

Full text:
  I had heard that love makes garbage a gold too. I saw it today. You didn't make me die like this, my friend. I'll have to live, man. The one who gave my life to my life, when did she die? For whom are you living? Because of this fake Guru, I lost my own self. What? I've shut down my business. I told you that I've booked a place here. You'll find a lot of drivers in this city. Forgive me. How did you book a place here? Do it now, sir. My love story started in the streets. Now it's over. Nowadays, people make proposals on WhatsApp. And they break them. But Vishal also loves me like this. Please take care of him. He's my son. Kunkar. Kunkar, your father seems to be in trouble. Kunkar.. There's no one in the whole world without a car. Raviji, why are you doing this? I am just a passe

# Transcription(for 10 videos)

In [4]:
!pip install -q yt-dlp
!apt-get -y install ffmpeg
!pip install faster-whisper

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 58.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 50.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 87.0 MB/s eta 0:00:00:00:0100:01


In [6]:
import yt_dlp
import os
import time
import csv
from pathlib import Path
from faster_whisper import WhisperModel

# ── CONFIG ──────────────────────────────────────────────────
MODEL_SIZE   = "small"
DEVICE       = "cpu"
COMPUTE_TYPE = "int8"
LANGUAGE     = "en"
CSV_PATH     = "/kaggle/working/transcriptions.csv"

VIDEO_IDS = [
    "R4Y7JIQlv20",
    "NJea386275c",
    "2TVXi_9Bvlg",
    "6zxnoT3XBu0",
    "LSIOcCcEVaE",
    "icoOOiSlKK0",
    "27FswS3KESk",
    "U4FAqwkn-pc",
    "2rcKpY-4QBI",
    "oYpUZjxJOVg",
]

# ── HELPERS ──────────────────────────────────────────────────
def load_model():
    print(f"\nLoading Whisper {MODEL_SIZE} on {DEVICE}...")
    start = time.time()
    model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
    print(f"Model loaded in {time.time()-start:.1f}s")
    return model

def download_audio(video_id, out_path="/kaggle/working/audio.mp3"):
    """Download audio for a given video ID, overwriting the same file each time."""
    # Remove old file if exists
    if os.path.exists(out_path):
        os.remove(out_path)

    video_url = f"https://youtube.com/watch?v={video_id}"
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': '/kaggle/working/audio.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
        }],
        'quiet': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])
    print(f"  Downloaded: {video_id}")

def transcribe_file(model, file_path):
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    start = time.time()
    segments, info = model.transcribe(
        str(file_path),
        language=LANGUAGE,
        beam_size=5,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500)
    )

    #all_segments = []
    full_text = []
    for segment in segments:
        '''all_segments.append({
            "start": round(segment.start, 2),
            "end":   round(segment.end, 2),
            "text":  segment.text.strip()
        })'''
        full_text.append(segment.text.strip())

    return {
        "text":       " ".join(full_text),
        #"segments":   all_segments,
        "language":   info.language,
        "duration":   round(info.duration, 2),
        "time_taken": round(time.time() - start, 2)
    }

def init_csv(csv_path):
    """Create CSV with headers if it doesn't already exist."""
    if not os.path.exists(csv_path):
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=[
                "video_id", "transcription"
            ])
            writer.writeheader()
        print(f"CSV created: {csv_path}")
    else:
        print(f"Appending to existing CSV: {csv_path}")

def write_to_csv(csv_path, row: dict):
    """Append a single row to the CSV."""
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "video_id", "transcription"
        ])
        writer.writerow(row)

# ── MAIN LOOP ────────────────────────────────────────────────
if __name__ == "__main__":
    init_csv(CSV_PATH)
    model = load_model()

    for i, video_id in enumerate(VIDEO_IDS, 1):
        print(f"\n[{i}/{len(VIDEO_IDS)}] Processing: {video_id}")
        try:
            download_audio(video_id)
            result = transcribe_file(model, "/kaggle/working/audio.mp3")

            write_to_csv(CSV_PATH, {
                "video_id":    video_id,
                "transcription": result["text"],
               })
            print(f"  ✓ Done — {result['duration']}s audio, {result['time_taken']}s to transcribe")

        except Exception as e:
            print(f"  ✗ Failed: {e}")

    print(f"\n All done! Results saved to: {CSV_PATH}")

Appending to existing CSV: /kaggle/working/transcriptions.csv

Loading Whisper small on cpu...
Model loaded in 1.0s

[1/10] Processing: R4Y7JIQlv20


  Downloaded: R4Y7JIQlv20                                  
  ✓ Done — 78.67s audio, 37.27s to transcribe

[2/10] Processing: NJea386275c


  Downloaded: NJea386275c                                  
  ✓ Done — 120.34s audio, 32.54s to transcribe

[3/10] Processing: 2TVXi_9Bvlg


  Downloaded: 2TVXi_9Bvlg                                  
  ✓ Done — 176.18s audio, 70.38s to transcribe

[4/10] Processing: 6zxnoT3XBu0


  Downloaded: 6zxnoT3XBu0                                  
  ✓ Done — 104.07s audio, 20.83s to transcribe

[5/10] Processing: LSIOcCcEVaE


...

  Downloaded: U4FAqwkn-pc                                  
  ✓ Done — 696.04s audio, 16.95s to transcribe

[9/10] Processing: 2rcKpY-4QBI


ERROR: [youtube] 2rcKpY-4QBI: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


  ✗ Failed: ERROR: [youtube] 2rcKpY-4QBI: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

[10/10] Processing: oYpUZjxJOVg


ERROR: [youtube] oYpUZjxJOVg: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


  ✗ Failed: ERROR: [youtube] oYpUZjxJOVg: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

 All done! Results saved to: /kaggle/working/transcriptions.csv


# Main Implementation for transcription

In [1]:
import pandas as pd, json, os

# Check what files are available
for f in os.listdir('/kaggle/input/datasets/rsrishav/youtube-trending-video-dataset'):
    print(f)

MX_youtube_trending_data.csv
GB_youtube_trending_data.csv
BR_category_id.json
FR_youtube_trending_data.csv
IN_youtube_trending_data.csv
MX_category_id.json
GB_category_id.json
US_category_id.json
BR_youtube_trending_data.csv
RU_youtube_trending_data.csv
IN_category_id.json
KR_category_id.json
DE_youtube_trending_data.csv
RU_category_id.json
US_youtube_trending_data.csv
JP_youtube_trending_data.csv
CA_category_id.json
DE_category_id.json
KR_youtube_trending_data.csv
JP_category_id.json
CA_youtube_trending_data.csv
FR_category_id.json


In [2]:
# Load US and GB only
us = pd.read_csv('/kaggle/input/datasets/rsrishav/youtube-trending-video-dataset/US_youtube_trending_data.csv')
gb = pd.read_csv('/kaggle/input/datasets/rsrishav/youtube-trending-video-dataset/GB_youtube_trending_data.csv')

with open('/kaggle/input/datasets/rsrishav/youtube-trending-video-dataset/US_category_id.json') as f:
    us_cats = json.load(f)
with open('/kaggle/input/datasets/rsrishav/youtube-trending-video-dataset/GB_category_id.json') as f:
    gb_cats = json.load(f)

# Map categories
def map_categories(df, cat_json):
    cat_map = {int(i['id']): i['snippet']['title'] for i in cat_json['items']}
    df['category'] = df['categoryId'].map(cat_map)
    return df

us = map_categories(us, us_cats)
gb = map_categories(gb, gb_cats)
us['country'] = 'US'
gb['country'] = 'GB'

df = pd.concat([us, gb], ignore_index=True)
df = df.drop_duplicates(subset='video_id')
df = df[df['view_count'] > 0].dropna(subset=['view_count','likes','comment_count'])

# Feature engineering
df['engagement_ratio'] = (df['likes'] + df['comment_count']) / df['view_count']

# Filter popular: top 30% views AND top 30% engagement
view_thresh = df['view_count'].quantile(0.70)
eng_thresh  = df['engagement_ratio'].quantile(0.70)

popular = df[(df['view_count'] >= view_thresh) & 
             (df['engagement_ratio'] >= eng_thresh)]

# Balanced top 500 across categories
top500 = (
    popular
    .sort_values('engagement_ratio', ascending=False)
    .groupby('category', group_keys=False)
    .apply(lambda x: x.nlargest(50, 'engagement_ratio'))
    .drop_duplicates(subset='video_id')
    .head(500)
    .reset_index(drop=True)
)

print(f"Popular videos : {len(popular)}")
print(f"Final selection: {len(top500)}")
print(top500['category'].value_counts())

# Export as output — you can download this single small file
top500.to_csv('top500_popular.csv', index=False)
df.to_csv('trending_master.csv', index=False)

/tmp/ipykernel_55/4290332670.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.nlargest(50, 'engagement_ratio'))


Popular videos : 6567
Final selection: 500
category
Comedy                   50
Entertainment            50
Education                50
Film & Animation         50
Gaming                   50
Music                    50
Howto & Style            50
People & Blogs           50
Autos & Vehicles         47
Science & Technology     30
News & Politics          11
Nonprofits & Activism     6
Pets & Animals            6
Name: count, dtype: int64


In [1]:
!pip install -q yt-dlp
!apt-get -y install ffmpeg
!pip install faster-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.3 MB/s eta 0:00:00a 0:00:01
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 58.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 51.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 66.0 MB/s eta 0:00:00:00:0100:01


In [3]:
import yt_dlp
import os
import time
import csv
import pandas as pd, json, os
from pathlib import Path
from faster_whisper import WhisperModel

# ── CONFIG ──────────────────────────────────────────────────
MODEL_SIZE   = "small"
DEVICE       = "cuda"
COMPUTE_TYPE = "int8"
LANGUAGE     = "en"
INPUT_CSV     = "/kaggle/input/datasets/sparshamp/top500-popular-videoids/top500_popular.csv"
CSV_PATH     = "/kaggle/working/transcriptions.csv"
START_FROM = 110


'''VIDEO_IDS = [
    "R4Y7JIQlv20",
    "NJea386275c",
    "2TVXi_9Bvlg",
    "6zxnoT3XBu0",
    "LSIOcCcEVaE",
    "icoOOiSlKK0",
    "27FswS3KESk",
    "U4FAqwkn-pc",
    "2rcKpY-4QBI",
    "oYpUZjxJOVg",
]'''

top500    = pd.read_csv(INPUT_CSV)
VIDEO_IDS = top500["video_id"].tolist()[START_FROM:]
print(f"Starting from index {START_FROM}, Total videos to process: {len(VIDEO_IDS)}")

# ── HELPERS ──────────────────────────────────────────────────
def load_model():
    print(f"\nLoading Whisper {MODEL_SIZE} on {DEVICE}...")
    start = time.time()
    model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
    print(f"Model loaded in {time.time()-start:.1f}s")
    return model

def download_audio(video_id, out_path="/kaggle/working/audio.mp3"):
    """Download audio for a given video ID, overwriting the same file each time."""
    # Remove old file if exists
    if os.path.exists(out_path):
        os.remove(out_path)

    video_url = f"https://youtube.com/watch?v={video_id}"
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': '/kaggle/working/audio.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
        }],
        'quiet': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])
    print(f"  Downloaded: {video_id}")

def transcribe_file(model, file_path):
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    start = time.time()
    segments, info = model.transcribe(
        str(file_path),
        language=LANGUAGE,
        beam_size=5,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500)
    )

    #all_segments = []
    full_text = []
    for segment in segments:
        '''all_segments.append({
            "start": round(segment.start, 2),
            "end":   round(segment.end, 2),
            "text":  segment.text.strip()
        })'''
        full_text.append(segment.text.strip())

    return {
        "text":       " ".join(full_text),
        #"segments":   all_segments,
        "language":   info.language,
        "duration":   round(info.duration, 2),
        "time_taken": round(time.time() - start, 2)
    }

def init_csv(csv_path):
    """Create CSV with headers if it doesn't already exist."""
    if not os.path.exists(csv_path):
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=[
                "video_id", "transcription"
            ])
            writer.writeheader()
        print(f"CSV created: {csv_path}")
    else:
        print(f"Appending to existing CSV: {csv_path}")

def write_to_csv(csv_path, row: dict):
    """Append a single row to the CSV."""
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "video_id", "transcription"
        ])
        writer.writerow(row)

# ── MAIN LOOP ────────────────────────────────────────────────
if __name__ == "__main__":
    init_csv(CSV_PATH)
    model = load_model()

    for i, video_id in enumerate(VIDEO_IDS, START_FROM + 1):
        print(f"\n[{i}/{len(VIDEO_IDS)}] Processing: {video_id}")
        try:
            download_audio(video_id)
            result = transcribe_file(model, "/kaggle/working/audio.mp3")

            write_to_csv(CSV_PATH, {
                "video_id":    video_id,
                "transcription": result["text"],
               })
            print(f"  ✓ Done — {result['duration']}s audio, {result['time_taken']}s to transcribe")

        except Exception as e:
            print(f"  ✗ Failed: {e}")

    print(f"\n All done! Results saved to: {CSV_PATH}")

Starting from index 110, Total videos to process: 390
CSV created: /kaggle/working/transcriptions.csv

Loading Whisper small on cuda...
Model loaded in 4.2s

[111/390] Processing: 7DKv5H5Frt0


  Downloaded: 7DKv5H5Frt0                                  
  ✓ Done — 490.29s audio, 10.85s to transcribe

[112/390] Processing: CYUDZGwUzXE


  Downloaded: CYUDZGwUzXE                                  
  ✓ Done — 775.59s audio, 33.37s to transcribe

[113/390] Processing: YIWV5fSaUB8


  Downloaded: YIWV5fSaUB8                                  
  ✓ Done — 1273.68s audio, 37.99s to transcribe

[114/390] Processing: TfVYxnhuEdU


  Downloaded: TfVYxnhuEdU                                  
  ✓ Done — 494.55s audio, 21.94s to transcribe

[115/390] Processing: 1x-i9z617z4


...

  ✗ Failed: ERROR: [youtube] mOFOjmws4hk: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

[499/390] Processing: rdKTjPDcTM0


ERROR: [youtube] rdKTjPDcTM0: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


  ✗ Failed: ERROR: [youtube] rdKTjPDcTM0: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

[500/390] Processing: cqidD7kVnxY


ERROR: [youtube] cqidD7kVnxY: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


  ✗ Failed: ERROR: [youtube] cqidD7kVnxY: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

 All done! Results saved to: /kaggle/working/transcriptions.csv
